In [1]:
# Spec P4 and 14: make all rebuilds, runs the Tier 1 grid and the RQ1 anchors, and builds F1-F6 and T1-T4,
# on AC power with the lid open and nothing else running. Notebook 16 bounds it at about 8.9 h of adapter time.
import json, subprocess, sys, time
from collections import Counter, defaultdict
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, validate

ALL = "all-20261112"
assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
t0 = time.monotonic()
r = subprocess.run(["make", "all", f"ALL={ALL}"], cwd=repo, capture_output=True, text=True)
print(f"make all: exit {r.returncode}, {(time.monotonic() - t0) / 3600:.2f} h wall")
print("\n".join(l for l in r.stderr.splitlines() if l.startswith(("all-", "stopped", "skipped"))))
runs = {tier: repo / "results/raw" / f"{ALL}_{tier}" for tier in ("tier1", "anchors")}
for tier, d in runs.items():
    meta = json.loads((d / "meta.json").read_text())
    print(tier, {k: meta[k] for k in ("git_sha_harness", "platform", "tier", "started")})
    assert meta["git_sha_harness"] and not meta["git_sha_harness"].endswith("-dirty"), f"{tier} ran on a dirty tree"
assert r.returncode == 0, r.stderr[-3000:]

make all: exit 0, 3.60 h wall
all-20261112_tier1: 198 of 206 done, 8 to run
all-20261112_tier1: {'timeout': 1, 'oom': 1, 'ok': 6}
all-20261112_anchors: 0 of 6 done, 6 to run
all-20261112_anchors: {'ok': 6}
skipped F8_session: no session comparison; pass --session results/raw/<run_id>/session after session_check.py
skipped F9_emulated_network: no emulated-link rows; pass --netem results/raw/<run_id>/netem after netem.py
skipped F10_load_latency: no load-generator rows; pass --loadgen results/raw/<run_id>/loadgen after harness/loadgen.py
skipped F11_cores: no load-generator rows; pass --loadgen results/raw/<run_id>/loadgen after harness/loadgen.py
tier1 {'git_sha_harness': 'e3eaf258a36c762d5b5d5d5b0caa553ff037b550', 'platform': 'arm64-darwin-Apple M1 Pro', 'tier': 'tier1', 'started': '2026-11-12T09:06:36.226877+00:00'}
anchors {'git_sha_harness': 'aef20f5cc8438a56265de621c1d2b2a6c620081d', 'platform': 'arm64-darwin-Apple M1 Pro', 'tier': 'anchors', 'started': '2026-11-12T21:31:12.663936+

In [2]:
# Every (cell, phase) of the Tier 1 grid: final status per arm, wrong answers, schema errors and peak footprint.
# A cell that ran out of memory, timed out or crashed is a result (spec 9), recorded with its status.
platform = json.loads((runs["tier1"] / "meta.json").read_text())["platform"]
by_arm, problems = defaultdict(Counter), []
for f in sorted(runs["tier1"].glob("*_*_*_*.jsonl")):
    rows = [json.loads(l) for l in f.read_text().splitlines()]
    status = sorted({x["status"] for x in rows})
    by_arm[f.stem.split("_")[0]][",".join(status)] += 1
    probs = [str(p) for x in rows for p in validate.check_row(x, reference_platform=platform) if p.severity == "error"]
    wrong = sum(x.get("correct") is False for x in rows if x.get("phase") == "online" and x["status"] == "ok")
    if probs or wrong:
        problems.append((f.stem, wrong, probs[:2]))
for arm, c in by_arm.items():
    print(f"{arm:7s} {dict(c)}")
print("not ok:", sorted(f.stem for f in runs["tier1"].glob("*_*_*_*.jsonl")
                       if any(json.loads(l)["status"] != "ok" for l in f.read_text().splitlines())))
print("wrong answers or schema errors:", problems)
assert not problems

M0      {'ok': 32}
M1      {'ok': 27, 'ok,oom': 1, 'oom': 4}
S0-cpp  {'ok': 32}
S0-go   {'ok': 32}
S1      {'ok': 28, 'oom': 4}
S2      {'ok': 24, 'oom': 8}
S3      {'ok': 6, 'ok,timeout': 1, 'oom': 1}
S4      {'ok': 6}
not ok: ['M1_128MiB_32B_offline', 'M1_1GiB_256B_offline', 'M1_1GiB_256B_online', 'M1_1GiB_32B_offline', 'M1_1GiB_32B_online', 'S1_1GiB_1KiB_offline', 'S1_1GiB_256B_offline', 'S1_1GiB_32B_offline', 'S1_1GiB_8KiB_offline', 'S2_1GiB_1KiB_offline', 'S2_1GiB_1KiB_online', 'S2_1GiB_256B_offline', 'S2_1GiB_256B_online', 'S2_1GiB_32B_offline', 'S2_1GiB_32B_online', 'S2_1GiB_8KiB_offline', 'S2_1GiB_8KiB_online', 'S3_1GiB_1bit_offline', 'S3_1GiB_1bit_online']
wrong answers or schema errors: []


In [3]:
# The Tier 1 figures and tables that make all built from the two runs.
made = sorted(p.name for p in (repo / "results/figures").iterdir() if p.is_file())
need = [f"F{i}_" for i in range(1, 7)] + [f"T{i}_" for i in range(1, 5)]
print(made)
print("missing:", [n for n in need if not any(m.startswith(n) for m in made)])

['F1_reproduction.pdf', 'F1_reproduction.png', 'F2_overhead_heatmaps.pdf', 'F2_overhead_heatmaps.png', 'F3_absolute_cost.pdf', 'F3_absolute_cost.png', 'F4_amortised.pdf', 'F4_amortised.png', 'F5_network.pdf', 'F5_network.png', 'F6_breakdown.pdf', 'F6_breakdown.png', 'T1_machine.csv', 'T1_machine.md', 'T2_scheme_params.csv', 'T2_scheme_params.md', 'T3_reproduction.csv', 'T3_reproduction.md', 'T4_overheads.csv', 'T4_overheads.md']
missing: []
